# Job Shop Scheduling con Algoritmos Genéticos

Cuaderno listo para **Google Colab** o **JupyterLab**.

Incluye:
- representación por secuencia de trabajos,
- decodificación a cronograma válido,
- selección por torneo,
- cruce adaptado,
- mutación por intercambio,
- gráfico de convergencia,
- diagrama de Gantt.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt


## Datos del problema


In [ ]:
trabajos = [
    [(0, 3), (1, 2), (2, 2)],
    [(0, 2), (2, 1), (1, 4)],
    [(1, 4), (2, 3), (0, 1)]
]

n_trabajos = len(trabajos)
n_maquinas = 3
ops_por_trabajo = [len(t) for t in trabajos]
longitud_cromosoma = sum(ops_por_trabajo)

tam_poblacion = 80
n_generaciones = 120
prob_cruce = 0.85
prob_mutacion = 0.10
tam_torneo = 3

random.seed(42)
np.random.seed(42)


## Representación de individuos


In [ ]:
base_cromosoma = []
for job_id, cant_ops in enumerate(ops_por_trabajo):
    base_cromosoma.extend([job_id] * cant_ops)

def crear_individuo():
    individuo = base_cromosoma.copy()
    random.shuffle(individuo)
    return individuo


## Decodificación y fitness


In [ ]:
def decodificar(individuo):
    siguiente_op = [0] * n_trabajos
    disponible_maquina = [0] * n_maquinas
    fin_trabajo = [0] * n_trabajos
    agenda = []

    for job_id in individuo:
        op_idx = siguiente_op[job_id]
        maquina, duracion = trabajos[job_id][op_idx]

        inicio = max(fin_trabajo[job_id], disponible_maquina[maquina])
        fin = inicio + duracion

        agenda.append({
            'job': job_id,
            'op': op_idx,
            'machine': maquina,
            'start': inicio,
            'end': fin,
            'duration': duracion
        })

        fin_trabajo[job_id] = fin
        disponible_maquina[maquina] = fin
        siguiente_op[job_id] += 1

    makespan = max(item['end'] for item in agenda)
    return agenda, makespan

def fitness(individuo):
    _, makespan = decodificar(individuo)
    return -makespan


## Operadores genéticos


In [ ]:
def seleccion_torneo(poblacion):
    candidatos = random.sample(poblacion, tam_torneo)
    return max(candidatos, key=fitness).copy()

def reparar(individuo):
    esperado = {j: ops_por_trabajo[j] for j in range(n_trabajos)}
    actual = {j: individuo.count(j) for j in range(n_trabajos)}
    faltantes = []
    sobrantes_idx = []

    for j in range(n_trabajos):
        if actual[j] < esperado[j]:
            faltantes.extend([j] * (esperado[j] - actual[j]))
        elif actual[j] > esperado[j]:
            exceso = actual[j] - esperado[j]
            encontrados = 0
            for idx, val in enumerate(individuo):
                if val == j and encontrados < exceso:
                    sobrantes_idx.append(idx)
                    encontrados += 1

    random.shuffle(faltantes)
    for idx, nuevo_val in zip(sobrantes_idx, faltantes):
        individuo[idx] = nuevo_val
    return individuo

def cruce_orden_multiconjunto(padre1, padre2):
    if random.random() >= prob_cruce:
        return padre1.copy(), padre2.copy()

    a, b = sorted(random.sample(range(len(padre1)), 2))
    hijo1 = [-1] * len(padre1)
    hijo2 = [-1] * len(padre2)
    hijo1[a:b] = padre1[a:b]
    hijo2[a:b] = padre2[a:b]

    def completar(hijo, otro_padre, segmento):
        usados = {j: segmento.count(j) for j in range(n_trabajos)}
        requerido = {j: ops_por_trabajo[j] for j in range(n_trabajos)}
        pos_libres = [i for i, x in enumerate(hijo) if x == -1]
        carga = []
        for gen in otro_padre:
            if usados.get(gen, 0) < requerido[gen]:
                carga.append(gen)
                usados[gen] = usados.get(gen, 0) + 1
        for pos, gen in zip(pos_libres, carga):
            hijo[pos] = gen
        return hijo

    hijo1 = completar(hijo1, padre2, padre1[a:b])
    hijo2 = completar(hijo2, padre1, padre2[a:b])
    return reparar(hijo1), reparar(hijo2)

def mutar_intercambio(individuo):
    mutado = individuo.copy()
    if random.random() < prob_mutacion:
        i, j = random.sample(range(len(mutado)), 2)
        mutado[i], mutado[j] = mutado[j], mutado[i]
    return mutado


## Ejecución del AG


In [ ]:
poblacion = [crear_individuo() for _ in range(tam_poblacion)]
mejores_fitness = []
promedios_fitness = []
mejor_individuo_global = None
mejor_fit_global = -float('inf')

for generacion in range(n_generaciones):
    nueva_poblacion = []
    while len(nueva_poblacion) < tam_poblacion:
        padre1 = seleccion_torneo(poblacion)
        padre2 = seleccion_torneo(poblacion)
        hijo1, hijo2 = cruce_orden_multiconjunto(padre1, padre2)
        hijo1 = mutar_intercambio(hijo1)
        hijo2 = mutar_intercambio(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < tam_poblacion:
            nueva_poblacion.append(hijo2)
    poblacion = nueva_poblacion
    fitness_poblacion = [fitness(ind) for ind in poblacion]
    mejor_generacion = max(poblacion, key=fitness)
    mejor_fit = fitness(mejor_generacion)
    promedio_fit = np.mean(fitness_poblacion)
    mejores_fitness.append(-mejor_fit)
    promedios_fitness.append(-promedio_fit)
    if mejor_fit > mejor_fit_global:
        mejor_fit_global = mejor_fit
        mejor_individuo_global = mejor_generacion.copy()

agenda_final, makespan_final = decodificar(mejor_individuo_global)
print('Mejor cromosoma encontrado:', mejor_individuo_global)
print('Makespan final:', makespan_final)
for op in agenda_final:
    print(op)


## Gráfico de convergencia


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_fitness, label='Mejor makespan')
plt.plot(promedios_fitness, label='Makespan promedio')
plt.xlabel('Generación')
plt.ylabel('Makespan')
plt.title('Evolución del algoritmo genético para JSSP')
plt.legend()
plt.grid(True)
plt.show()


## Diagrama de Gantt


In [ ]:
plt.figure(figsize=(10, 5))
colores = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

for item in agenda_final:
    maquina = item['machine']
    inicio = item['start']
    duracion = item['duration']
    job = item['job']
    op = item['op']
    plt.barh(maquina, duracion, left=inicio, height=0.5, color=colores[job], edgecolor='black')
    plt.text(inicio + duracion / 2, maquina, f'J{job}-O{op}', ha='center', va='center', color='white', fontsize=9)

plt.yticks(range(n_maquinas), [f'M{m}' for m in range(n_maquinas)])
plt.xlabel('Tiempo')
plt.ylabel('Máquina')
plt.title('Diagrama de Gantt de la mejor solución')
plt.grid(axis='x')
plt.show()


## Ejercicio sugerido

Modificar:
- la instancia de trabajos,
- el tamaño de la población,
- la cantidad de generaciones,
- la probabilidad de mutación,

y comparar cómo cambia el makespan final.
